# 🚀 AI Vision Pipeline (Colab 최적화 버전)
Google Colab 환경에서 바로 실행 가능한 버전입니다.

## 실행 순서
1. Google Drive 마운트
2. 경로 설정
3. 데이터 병합 (COCO 생성)
4. EDA
5. YOLO 변환
6. 데이터 증강


## 0️⃣ Google Drive 마운트

## ⚙️ 필수 라이브러리 설치

In [1]:
!pip install albumentations pillow matplotlib


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 0️⃣ 경로 설정

In [3]:
# 🔥 여기를 본인 환경에 맞게 수정하세요
BASE_DIR = '/content/drive/MyDrive/ai_project'
ZIP_PATH = f'{BASE_DIR}/ai10-level1-project.zip'
OUTPUT_DIR = f'{BASE_DIR}/output'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('BASE_DIR:', BASE_DIR)


BASE_DIR: /content/drive/MyDrive/ai_project


## 1️⃣ 데이터 병합 (COCO 생성)

In [6]:
import json
import zipfile
from pathlib import Path
from collections import defaultdict


# 데이터 로드
ZIP_PATH   = Path(ZIP_PATH)
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 파일 목록 확인
print("=" * 55)
print("1단계: ZIP 파일 읽는 중...")
print("=" * 55)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    all_files = z.namelist()

train_jsons = [
    f for f in all_files
    if f.endswith(".json") and "train_annotations" in f
]
train_pngs = [
    f for f in all_files
    if f.endswith(".png") and "train_images" in f
]

print(f"  발견한 train JSON : {len(train_jsons)}개")
print(f"  발견한 train PNG  : {len(train_pngs)}개")


# 카테고리 수집 및 순차적으로 ID 부여 후 매핑
print("\n" + "=" * 55)
print("2단계: 카테고리 수집 & ID 매핑 생성 중...")
print("=" * 55)

# 카테고리명 → dl_idx(원본 숫자 ID) 수집
#   JSON 구조: categories[0].name = 약품명, categories[0].id = dl_idx
name_to_dlid: dict[str, int] = {}   # 약품명 → 원본 dl_idx
dlid_to_name: dict[int, str] = {}   # 원본 dl_idx → 약품명

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for jf in train_jsons:
        with z.open(jf) as f:
            data = json.load(f)
        for cat in data.get("categories", []):
            name = cat.get("name", "").strip()
            cid  = cat.get("id")
            if name and cid is not None:
                name_to_dlid[name] = cid
                dlid_to_name[cid]  = name

# 카테고리명 기준으로 정렬 후 1-based 순차 ID 부여
sorted_names = sorted(name_to_dlid.keys())
name_to_newid: dict[str, int] = {
    name: idx + 1 for idx, name in enumerate(sorted_names)
}
newid_to_name: dict[int, str] = {v: k for k, v in name_to_newid.items()}
dlid_to_newid: dict[int, int] = {
    name_to_dlid[name]: name_to_newid[name]
    for name in sorted_names
    if name in name_to_dlid
}

print(f"  고유 카테고리 수  : {len(sorted_names)}개")
print(f"  ID 범위           : 1 ~ {len(sorted_names)}")

# 매핑 테이블 저장
mapping = {
    "description": "category_name ↔ category_id 매핑 테이블",
    "total_categories": len(sorted_names),
    "name_to_id": name_to_newid,
    "id_to_name": {str(k): v for k, v in newid_to_name.items()},
    "original_dlid_to_new_id": {str(k): v for k, v in dlid_to_newid.items()},
}
mapping_path = OUTPUT_DIR / "category_mapping.json"
with open(mapping_path, "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)
print(f"  매핑 테이블 저장  : {mapping_path}")


# 이미지 파일명 및 이미지 ID 매핑
print("\n" + "=" * 55)
print("3단계: 이미지 ID 매핑 생성 중...")
print("=" * 55)

# train_images 폴더 안의 PNG 파일명으로 image_id 결정
# 파일명 예: K-001900-016548-019607-029451_0_2_0_2_70_000_200.png
# image_id = 파일명의 숫자 부분 (파일명 자체를 hash 또는 순번으로)

filename_to_imageid: dict[str, int] = {}
imageid_to_filename: dict[int, str] = {}

for idx, png_path in enumerate(sorted(train_pngs), start=1):
    fname = Path(png_path).name   # 예: K-001900-..._200.png
    filename_to_imageid[fname] = idx
    imageid_to_filename[idx]   = fname

print(f"  총 이미지 수      : {len(filename_to_imageid)}개")


#전체 JSON 파싱 → COCO 통합 구조
print("\n" + "=" * 55)
print("4단계: JSON 763개 파싱 & COCO 통합 JSON 조립 중...")
print("=" * 55)

coco_images:      list[dict] = []
coco_annotations: list[dict] = []

seen_image_ids:  set[int] = set()
annotation_id_counter = 1
skip_count   = 0
bbox_invalid = 0

# 이미 수집한 image 정보를 JSON에서도 읽어둠
image_id_info: dict[int, dict] = {}  # image_id → {width, height, file_name}

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for jf in train_jsons:
        with z.open(jf) as f:
            data = json.load(f)

        # 이미지 정보
        images_in_json = data.get("images", [])
        if not images_in_json:
            skip_count += 1
            continue

        img_info  = images_in_json[0]
        file_name = img_info.get("file_name", "")

        # image_id 결정: 파일명으로 찾거나 없으면 신규 부여
        fname_only = Path(file_name).name
        if fname_only in filename_to_imageid:
            image_id = filename_to_imageid[fname_only]
        else:
            # 파일명이 다를 경우 순번으로 처리
            image_id = len(filename_to_imageid) + len(seen_image_ids) + 1
            filename_to_imageid[fname_only] = image_id

        # 이미지 중복 등록 방지
        if image_id not in seen_image_ids:
            seen_image_ids.add(image_id)
            image_id_info[image_id] = {
                "id"       : image_id,
                "file_name": fname_only,
                "width"    : img_info.get("width", 0),
                "height"   : img_info.get("height", 0),
            }

        # 어노테이션
        for ann in data.get("annotations", []):
            bbox = ann.get("bbox")

            # 유효성 검사: bbox 존재 & 길이 4
            if not isinstance(bbox, list) or len(bbox) != 4:
                bbox_invalid += 1
                continue

            # bbox 값이 모두 0인 경우 제외 (빈 bbox)
            if all(v == 0 for v in bbox):
                bbox_invalid += 1
                continue

            # category_id 변환: 원본 dl_idx → 새 순차 ID
            orig_cat_id = ann.get("category_id")
            new_cat_id  = dlid_to_newid.get(orig_cat_id)
            if new_cat_id is None:
                # 카테고리명 기반 재시도
                cats = data.get("categories", [])
                if cats:
                    cat_name   = cats[0].get("name", "").strip()
                    new_cat_id = name_to_newid.get(cat_name)
            if new_cat_id is None:
                skip_count += 1
                continue

            # bbox: [x, y, w, h] → 면적 계산
            x, y, w, h = [float(v) for v in bbox]
            area = w * h

            coco_annotations.append({
                "id"         : annotation_id_counter,
                "image_id"   : image_id,
                "category_id": new_cat_id,
                "bbox"       : [x, y, w, h],
                "area"       : round(area, 2),
                "iscrowd"    : 0,
                "segmentation": [],
            })
            annotation_id_counter += 1

# 이미지 리스트 정렬
coco_images = sorted(image_id_info.values(), key=lambda x: x["id"])

print(f"  처리된 이미지 수  : {len(coco_images)}개")
print(f"  유효 어노테이션   : {len(coco_annotations)}개")
print(f"  무효/스킵 수      : {skip_count + bbox_invalid}개")


# 5단계: COCO categories 리스트 생성
coco_categories = [
    {
        "id"           : new_id,
        "name"         : name,
        "supercategory": "pill",
    }
    for name, new_id in sorted(name_to_newid.items(), key=lambda x: x[1])
]


# 6단계: 통합 COCO JSON 저장
print("\n" + "=" * 55)
print("5단계: 통합 COCO JSON 저장 중...")
print("=" * 55)

coco_output = {
    "info": {
        "description": "알약 탐지 프로젝트 통합 어노테이션",
        "version"    : "1.0",
        "total_images"     : len(coco_images),
        "total_annotations": len(coco_annotations),
        "total_categories" : len(coco_categories),
    },
    "images"     : coco_images,
    "annotations": coco_annotations,
    "categories" : coco_categories,
}

output_path = OUTPUT_DIR / "train_coco.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(coco_output, f, ensure_ascii=False, indent=2)
print(f"  저장 완료         : {output_path}")


# 7단계: 데이터 현황 요약 리포트
print("\n" + "=" * 55)
print("6단계: 데이터 현황 리포트 생성 중...")
print("=" * 55)

# 클래스별 어노테이션 수 집계
class_ann_count: dict[int, int] = defaultdict(int)
for ann in coco_annotations:
    class_ann_count[ann["category_id"]] += 1

# 이미지당 알약 수 집계
img_pill_count: dict[int, int] = defaultdict(int)
for ann in coco_annotations:
    img_pill_count[ann["image_id"]] += 1

pill_counts   = list(img_pill_count.values())
avg_per_image = sum(pill_counts) / len(pill_counts) if pill_counts else 0
dist          = defaultdict(int)
for c in pill_counts:
    dist[c] += 1

report_lines = [
    "=" * 55,
    "  알약 탐지 데이터셋 현황 요약",
    "=" * 55,
    f"  Train 이미지 수        : {len(coco_images)}장",
    f"  Train 어노테이션 수    : {len(coco_annotations)}개",
    f"  카테고리(약품) 수      : {len(coco_categories)}종",
    f"  이미지당 평균 알약 수  : {avg_per_image:.2f}개",
    "",
    "  이미지당 알약 수 분포",
    *[f"    알약 {k}개짜리 이미지  : {v}장" for k, v in sorted(dist.items())],
    "",
    "  클래스별 어노테이션 수 (적은 순)",
    *[
        f"    [{new_id:>3}] {newid_to_name[new_id]:<35} : {class_ann_count[new_id]}개"
        for new_id in sorted(class_ann_count, key=lambda x: class_ann_count[x])
    ],
    "",
    f"  최소 클래스 어노테이션 : {min(class_ann_count.values())}개",
    f"  최대 클래스 어노테이션 : {max(class_ann_count.values())}개",
    f"  평균 클래스 어노테이션 : {sum(class_ann_count.values())/len(class_ann_count):.1f}개",
    "=" * 55,
]

report_text = "\n".join(report_lines)
print(report_text)

summary_path = OUTPUT_DIR / "dataset_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(report_text)
print(f"\n  리포트 저장       : {summary_path}")

print("\n" + "=" * 55)
print("  완료! 생성된 파일 목록")
print("=" * 55)
print(f"  1. {OUTPUT_DIR / 'train_coco.json'}")
print(f"  2. {OUTPUT_DIR / 'category_mapping.json'}")
print(f"  3. {OUTPUT_DIR / 'dataset_summary.txt'}")
print("=" * 55)

1단계: ZIP 파일 읽는 중...
  발견한 train JSON : 763개
  발견한 train PNG  : 232개

2단계: 카테고리 수집 & ID 매핑 생성 중...
  고유 카테고리 수  : 56개
  ID 범위           : 1 ~ 56
  매핑 테이블 저장  : /content/drive/MyDrive/ai_project/output/category_mapping.json

3단계: 이미지 ID 매핑 생성 중...
  총 이미지 수      : 232개

4단계: JSON 763개 파싱 & COCO 통합 JSON 조립 중...
  처리된 이미지 수  : 232개
  유효 어노테이션   : 763개
  무효/스킵 수      : 0개

5단계: 통합 COCO JSON 저장 중...
  저장 완료         : /content/drive/MyDrive/ai_project/output/train_coco.json

6단계: 데이터 현황 리포트 생성 중...
  알약 탐지 데이터셋 현황 요약
  Train 이미지 수        : 232장
  Train 어노테이션 수    : 763개
  카테고리(약품) 수      : 56종
  이미지당 평균 알약 수  : 3.29개

  이미지당 알약 수 분포
    알약 2개짜리 이미지  : 7장
    알약 3개짜리 이미지  : 151장
    알약 4개짜리 이미지  : 74장

  클래스별 어노테이션 수 (적은 순)
    [ 11] 레일라정                                : 3개
    [ 26] 신바로정                                : 3개
    [ 10] 라비에트정 20mg                          : 3개
    [ 39] 울트라셋이알서방정                           : 3개
    [  6] 놀텍정 10mg                            : 3개
    [  9] 동아가바펜틴정 8

## 2️⃣ EDA

In [9]:
import json
import zipfile
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import defaultdict

# 한글 폰트 설치 및 설정 (Colab 환경)
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm -rf ~/.cache/matplotlib

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 경로 설정
# ZIP_PATH와 OUTPUT_DIR은 이미 Path 객체일 수 있으므로, Path()로 한 번 더 감싸서 확실히 Path 객체로 만듭니다.
ZIP_PATH     = Path(ZIP_PATH)
OUTPUT_DIR   = Path(OUTPUT_DIR)
COCO_JSON    = OUTPUT_DIR / "train_coco.json"
MAPPING_JSON = OUTPUT_DIR / "category_mapping.json"
EDA_DIR      = OUTPUT_DIR / "eda"
EDA_DIR.mkdir(exist_ok=True)

print("=" * 60)
print("EDA 시작")
print("=" * 60)

# 데이터 로드
with open(COCO_JSON, encoding="utf-8") as f:
    coco = json.load(f)
with open(MAPPING_JSON, encoding="utf-8") as f:
    mapping = json.load(f)

images      = coco["images"]
annotations = coco["annotations"]
categories  = coco["categories"]
id_to_name  = {int(k): v for k, v in mapping["id_to_name"].items()}

print(f"  이미지 수      : {len(images)}장")
print(f"  어노테이션 수  : {len(annotations)}개")
print(f"  카테고리 수    : {len(categories)}종")

# 집계
# 클래스별 count
class_count = defaultdict(int)
for ann in annotations:
    class_count[ann["category_id"]] += 1

# 이미지당 알약 수
img_pill_count = defaultdict(int)
for ann in annotations:
    img_pill_count[ann["image_id"]] += 1

# bbox 크기 분포
bbox_areas  = [ann["area"] for ann in annotations]
bbox_widths = [ann["bbox"][2] for ann in annotations]
bbox_heights= [ann["bbox"][3] for ann in annotations]

# 이미지 크기
img_widths  = [img["width"]  for img in images]
img_heights = [img["height"] for img in images]

# 시각화 1: 클래스별 어노테이션 수
print("\n[1] 클래스별 분포 시각화...")
sorted_classes = sorted(class_count.items(), key=lambda x: x[1], reverse=True)
names  = [id_to_name.get(cid, str(cid))[:10] for cid, _ in sorted_classes]
counts = [cnt for _, cnt in sorted_classes]

fig, ax = plt.subplots(figsize=(18, 6))
colors = ['#E24B4A' if c <= 5 else '#EF9F27' if c <= 12 else '#1D9E75'
          for c in counts]
bars = ax.bar(range(len(names)), counts, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_xlabel("약품명")
ax.set_ylabel("어노테이션 수")
ax.set_title("클래스별 어노테이션 수 분포 (빨강=3~5개, 주황=6~12개, 초록=13개+)")
ax.axhline(y=np.mean(counts), color='gray', linestyle='--', alpha=0.7,
           label=f'평균 {np.mean(counts):.1f}개')
ax.legend()

# 막대 위 숫자
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(cnt), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(EDA_DIR / "01_class_distribution.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"  저장: {EDA_DIR / '01_class_distribution.png'}")

# 시각화 2: 이미지당 알약 수 분포
print("[2] 이미지당 알약 수 분포...")
pill_dist = defaultdict(int)
for cnt in img_pill_count.values():
    pill_dist[cnt] += 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 파이 차트
labels = [f"{k}개짜리\n({v}장)" for k, v in sorted(pill_dist.items())]
sizes  = [v for _, v in sorted(pill_dist.items())]
colors_pie = ['#85B7EB', '#1D9E75', '#EF9F27']
axes[0].pie(sizes, labels=labels, colors=colors_pie[:len(sizes)],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title("이미지당 알약 수 분포")

# 막대 차트
keys = sorted(pill_dist.keys())
axes[1].bar([str(k) + "개" for k in keys],
            [pill_dist[k] for k in keys],
            color=colors_pie[:len(keys)], edgecolor='white')
axes[1].set_xlabel("이미지당 알약 수")
axes[1].set_ylabel("이미지 수")
axes[1].set_title("이미지당 알약 수")
for i, (k, v) in enumerate([(k, pill_dist[k]) for k in keys]):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(EDA_DIR / "02_pills_per_image.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"  저장: {EDA_DIR / '02_pills_per_image.png'}")

# 시각화 3: bbox 크기 분포
print("[3] bbox 크기 분포...")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(bbox_widths,  bins=30, color='#378ADD', edgecolor='white', alpha=0.8)
axes[0].set_title("bbox 너비 분포")
axes[0].set_xlabel("픽셀")
axes[0].axvline(np.mean(bbox_widths), color='red', linestyle='--',
                label=f"평균 {np.mean(bbox_widths):.0f}px")
axes[0].legend()

axes[1].hist(bbox_heights, bins=30, color='#1D9E75', edgecolor='white', alpha=0.8)
axes[1].set_title("bbox 높이 분포")
axes[1].set_xlabel("픽셀")
axes[1].axvline(np.mean(bbox_heights), color='red', linestyle='--',
                label=f"평균 {np.mean(bbox_heights):.0f}px")
axes[1].legend()

axes[2].hist(bbox_areas,   bins=30, color='#EF9F27', edgecolor='white', alpha=0.8)
axes[2].set_title("bbox 면적 분포")
axes[2].set_xlabel("픽셀²")
axes[2].axvline(np.mean(bbox_areas), color='red', linestyle='--',
                label=f"평균 {np.mean(bbox_areas):.0f}px²")
axes[2].legend()

plt.tight_layout()
plt.savefig(EDA_DIR / "03_bbox_size_distribution.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"  저장: {EDA_DIR / '03_bbox_size_distribution.png'}")

# 시각화 4: bbox 위치 히트맵
print("[4] bbox 위치 히트맵...")
# 정규화된 중심점 수집
cx_list, cy_list = [], []
for ann in annotations:
    img_id = ann["image_id"]
    img_info = next((img for img in images if img["id"] == img_id), None)
    if img_info is None:
        continue
    x, y, w, h = ann["bbox"]
    cx = (x + w/2) / img_info["width"]
    cy = (y + h/2) / img_info["height"]
    cx_list.append(cx)
    cy_list.append(cy)

fig, ax = plt.subplots(figsize=(7, 6))
heatmap, xedges, yedges = np.histogram2d(cx_list, cy_list, bins=20,
                                          range=[[0,1],[0,1]])
im = ax.imshow(heatmap.T, origin='lower', cmap='YlOrRd',
               extent=[0, 1, 0, 1], aspect='auto')
plt.colorbar(im, ax=ax, label='bbox 빈도')
ax.set_xlabel("이미지 너비 방향 (0=왼쪽, 1=오른쪽)")
ax.set_ylabel("이미지 높이 방향 (0=위, 1=아래)")
ax.set_title("bbox 중심점 위치 히트맵\n(알약이 이미지 어디에 주로 위치하는가)")
plt.tight_layout()
plt.savefig(EDA_DIR / "04_bbox_heatmap.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"  저장: {EDA_DIR / '04_bbox_heatmap.png'}")

# 시각화 5: 샘플 이미지 + bbox 시각화
print("[5] 샘플 이미지 bbox 시각화 (3장)...")

# 이미지별 어노테이션 그룹핑
img_anns = defaultdict(list)
for ann in annotations:
    img_anns[ann["image_id"]].append(ann)

# 알약 4개짜리 이미지 우선 선택
target_imgs = [img for img in images
               if len(img_anns.get(img["id"], [])) == 4][:3]
if len(target_imgs) < 3:
    target_imgs = images[:3]

# 색상 팔레트
colors_bbox = ['#E24B4A', '#1D9E75', '#378ADD', '#EF9F27']

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    png_map = {Path(f).name: f for f in z.namelist()
               if f.endswith(".png") and "train_images" in f}

    fig, axes = plt.subplots(1, len(target_imgs),
                              figsize=(6 * len(target_imgs), 6))
    if len(target_imgs) == 1:
        axes = [axes]

    for ax, img_info in zip(axes, target_imgs):
        fname = img_info["file_name"]
        if fname not in png_map:
            continue

        import io
        from PIL import Image
        with z.open(png_map[fname]) as f:
            img_pil = Image.open(io.BytesIO(f.read())).convert("RGB")

        ax.imshow(img_pil)
        ax.axis('off')
        ax.set_title(f"{fname[:30]}\n"
                     f"({img_info['width']}×{img_info['height']})", fontsize=8)

        for i, ann in enumerate(img_anns.get(img_info["id"], [])):
            x, y, w, h = ann["bbox"]
            cat_name = id_to_name.get(ann["category_id"], "?")[:8]
            color = colors_bbox[i % len(colors_bbox)]
            rect = patches.Rectangle((x, y), w, h,
                                      linewidth=1.5, edgecolor=color,
                                      facecolor='none')
            ax.add_patch(rect)
            ax.text(x, max(0, y-3), name, color=color, fontsize=6,
                    bbox=dict(facecolor='white', alpha=0.6,
                              edgecolor='none', pad=1))

plt.suptitle("샘플 이미지 + bbox 시각화", fontsize=12)
plt.tight_layout()
plt.savefig(EDA_DIR / "05_sample_images.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"  저장: {EDA_DIR / '05_sample_images.png'}")

# EDA 요약 출력
print("\n" + "=" * 60)
print("EDA 요약")
print("=" * 60)
print(f"  이미지 크기 범위  : {min(img_widths)}~{max(img_widths)} × "
      f"{min(img_heights)}~{max(img_heights)}")
print(f"  bbox 너비 평균    : {np.mean(bbox_widths):.1f}px  "
      f"(min {min(bbox_widths):.0f} / max {max(bbox_widths):.0f})")
print(f"  bbox 높이 평균    : {np.mean(bbox_heights):.1f}px  "
      f"(min {min(bbox_heights):.0f} / max {max(bbox_heights):.0f})")
print(f"  이미지당 알약 수  : {dict(sorted(pill_dist.items()))}")
print(f"  클래스 불균형 비율: {max(counts)} / {min(counts)} = "
      f"{max(counts)/min(counts):.1f}배")
print(f"\n  EDA 결과 저장 위치: {EDA_DIR}")
print("=" * 60)


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 1s (9,555 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 118194 files and direc

/tmp/ipykernel_2032/2040210007.py:91: UserWarning: Glyph 51068 (\N{HANGUL SYLLABLE IL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:91: UserWarning: Glyph 50577 (\N{HANGUL SYLLABLE YANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:91: UserWarning: Glyph 54616 (\N{HANGUL SYLLABLE HA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:91: UserWarning: Glyph 51060 (\N{HANGUL SYLLABLE I}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:91: UserWarning: Glyph 53944 (\N{HANGUL SYLLABLE TEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:91: UserWarning: Glyph 47536 (\N{HANGUL SYLLABLE RIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:91: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans.
  plt.tight_la

  저장: /content/drive/MyDrive/ai_project/output/eda/01_class_distribution.png
[2] 이미지당 알약 수 분포...


/tmp/ipykernel_2032/2040210007.py:124: UserWarning: Glyph 51060 (\N{HANGUL SYLLABLE I}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "02_pills_per_image.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2032/2040210007.py:124: UserWarning: Glyph 48120 (\N{HANGUL SYLLABLE MI}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "02_pills_per_image.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2032/2040210007.py:124: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "02_pills_per_image.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2032/2040210007.py:124: UserWarning: Glyph 45817 (\N{HANGUL SYLLABLE DANG}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "02_pills_per_image.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2032/2040210007.py:124: UserWarning: Glyph 50508 (\N{HANGUL SYLLABLE AL}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "02_pills_per_image.png", dpi=150, bbox_

  저장: /content/drive/MyDrive/ai_project/output/eda/02_pills_per_image.png
[3] bbox 크기 분포...


...

  저장: /content/drive/MyDrive/ai_project/output/eda/03_bbox_size_distribution.png
[4] bbox 위치 히트맵...


/tmp/ipykernel_2032/2040210007.py:183: UserWarning: Glyph 45320 (\N{HANGUL SYLLABLE NEO}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "04_bbox_heatmap.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2032/2040210007.py:183: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "04_bbox_heatmap.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2032/2040210007.py:183: UserWarning: Glyph 50812 (\N{HANGUL SYLLABLE OEN}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "04_bbox_heatmap.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2032/2040210007.py:183: UserWarning: Glyph 51901 (\N{HANGUL SYLLABLE JJOG}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "04_bbox_heatmap.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2032/2040210007.py:183: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from font(s) DejaVu Sans.
  plt.savefig(EDA_DIR / "04_bbox_heatmap.png", dpi=150, bbox_inches='tight

  저장: /content/drive/MyDrive/ai_project/output/eda/04_bbox_heatmap.png
[5] 샘플 이미지 bbox 시각화 (3장)...


/tmp/ipykernel_2032/2040210007.py:241: UserWarning: Glyph 51320 (\N{HANGUL SYLLABLE JOL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:241: UserWarning: Glyph 47196 (\N{HANGUL SYLLABLE RO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:241: UserWarning: Glyph 54392 (\N{HANGUL SYLLABLE PU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:241: UserWarning: Glyph 53944 (\N{HANGUL SYLLABLE TEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:241: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:241: UserWarning: Glyph 49368 (\N{HANGUL SYLLABLE SAEM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/2040210007.py:241: UserWarning: Glyph 54540 (\N{HANGUL SYLLABLE PEUL}) missing from font(s) DejaVu Sans.
  pl

  저장: /content/drive/MyDrive/ai_project/output/eda/05_sample_images.png

EDA 요약
  이미지 크기 범위  : 976~976 × 1280~1280
  bbox 너비 평균    : 250.0px  (min 125 / max 529)
  bbox 높이 평균    : 265.8px  (min 126 / max 664)
  이미지당 알약 수  : {2: 7, 3: 151, 4: 74}
  클래스 불균형 비율: 153 / 3 = 51.0배

  EDA 결과 저장 위치: /content/drive/MyDrive/ai_project/output/eda


## 3️⃣ COCO → YOLO 변환

In [11]:
import json
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict

import random
random.seed(42)

# 경로 설정
ZIP_PATH      = Path(ZIP_PATH)
OUTPUT_DIR    = Path(OUTPUT_DIR)
COCO_JSON     = OUTPUT_DIR / "train_coco.json"
MAPPING_JSON  = OUTPUT_DIR / "category_mapping.json"
DATASET_DIR   = OUTPUT_DIR / "dataset"
VAL_RATIO     = 0.2   # 80/20 split

# 데이터셋 폴더 생성
for split in ["train", "val"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# COCO JSON 로드
print("=" * 55)
print("COCO JSON → YOLO 포맷 변환 시작")
print("=" * 55)

with open(COCO_JSON, encoding="utf-8") as f:
    coco = json.load(f)

with open(MAPPING_JSON, encoding="utf-8") as f:
    mapping = json.load(f)

# 카테고리 정보 (0-based 변환용)
id_to_name = {int(k): v for k, v in mapping["id_to_name"].items()}
# COCO category_id(1-based) → YOLO class_id(0-based)
coco_id_to_yolo = {cat_id: cat_id - 1 for cat_id in id_to_name}

# 이미지 정보 딕셔너리
images = {img["id"]: img for img in coco["images"]}

# 이미지별 어노테이션 그룹핑
img_anns: dict[int, list] = defaultdict(list)
for ann in coco["annotations"]:
    img_anns[ann["image_id"]].append(ann)

# Stratified Train/Val Split
# 각 클래스가 val에 최소 1개 이상 들어가도록 분할
print("\n[1] Stratified train/val split 진행 중...")

# 클래스별 이미지 ID 수집
class_to_imgs: dict[int, set] = defaultdict(set)
for ann in coco["annotations"]:
    class_to_imgs[ann["category_id"]].add(ann["image_id"])

all_img_ids = list(images.keys())
random.shuffle(all_img_ids)

# 전체 80/20 비율 기준으로 val 수 고정
n_total = len(all_img_ids)
n_val   = max(1, round(n_total * VAL_RATIO))   # 약 46장
n_train = n_total - n_val                       # 약 186장

# 클래스별로 val에 최소 1장 보장하면서 총 n_val을 넘지 않도록
val_img_ids: set[int] = set()

# 1단계: 소수 클래스(3개짜리)부터 val에 1장씩 배정
sorted_classes = sorted(class_to_imgs.items(), key=lambda x: len(x[1]))
for cat_id, img_ids in sorted_classes:
    if len(val_img_ids) >= n_val:
        break
    # 아직 val에 없는 이미지 중 1장만 배정
    candidates = list(img_ids - val_img_ids)
    if candidates:
        val_img_ids.add(random.choice(candidates))

# 2단계: 나머지 val 슬롯을 랜덤으로 채움 (총 n_val까지)
remaining = [i for i in all_img_ids if i not in val_img_ids]
random.shuffle(remaining)
still_need = n_val - len(val_img_ids)
val_img_ids.update(remaining[:still_need])

train_img_ids = set(all_img_ids) - val_img_ids

print(f"  Train 이미지: {len(train_img_ids)}장")
print(f"  Val   이미지: {len(val_img_ids)}장")

# YOLO txt 파일 생성 + 이미지 복사
print("\n[2] YOLO txt 라벨 파일 생성 중...")

# ZIP에서 이미지 추출
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    zip_png_map = {
        Path(f).name: f
        for f in z.namelist()
        if f.endswith(".png") and "train_images" in f
    }

    converted = 0
    skipped   = 0

    for img_id, img_info in images.items():
        fname  = img_info["file_name"]
        width  = img_info["width"]
        height = img_info["height"]

        split  = "train" if img_id in train_img_ids else "val"

        # 이미지 복사
        if fname in zip_png_map:
            img_dst = DATASET_DIR / "images" / split / fname
            if not img_dst.exists():
                with z.open(zip_png_map[fname]) as src:
                    img_dst.write_bytes(src.read())
        else:
            skipped += 1
            continue

        # YOLO txt 생성
        anns = img_anns.get(img_id, [])
        txt_dst = DATASET_DIR / "labels" / split / (Path(fname).stem + ".txt")

        lines = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            yolo_id = coco_id_to_yolo[ann["category_id"]]

            # COCO (x_min, y_min, w, h) → YOLO (cx, cy, w, h) 정규화
            cx = (x + w / 2) / width
            cy = (y + h / 2) / height
            nw = w / width
            nh = h / height

            # 범위 클램핑 (0~1 벗어나면 보정)
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            nw = max(0.0, min(1.0, nw))
            nh = max(0.0, min(1.0, nh))

            lines.append(f"{yolo_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        txt_dst.write_text("\n".join(lines), encoding="utf-8")
        converted += 1

print(f"  변환 완료: {converted}개")
print(f"  스킵 (이미지 없음): {skipped}개")

# pill.yaml 생성
print("\n[3] pill.yaml 생성 중...")

# 0-based 클래스명 목록 (정렬 순서 유지)
names_list = [id_to_name[i + 1] for i in range(len(id_to_name))]

yaml_content = f"""# 알약 탐지 프로젝트 — 데이터셋 설정
path: {DATASET_DIR.as_posix()}
train: images/train
val:   images/val

nc: {len(names_list)}

names:
"""
for i, name in enumerate(names_list):
    yaml_content += f"  {i}: {name}\n"

yaml_path = OUTPUT_DIR / "pill.yaml"
yaml_path.write_text(yaml_content, encoding="utf-8")
print(f"  저장: {yaml_path}")

# 클래스별 split 분포 검증
print("\n[4] Split 분포 검증...")

train_class_count: dict[int, int] = defaultdict(int)
val_class_count:   dict[int, int] = defaultdict(int)

for ann in coco["annotations"]:
    if ann["image_id"] in train_img_ids:
        train_class_count[ann["category_id"]] += 1
    else:
        val_class_count[ann["category_id"]] += 1

# val에 없는 클래스 확인
missing_in_val = [
    id_to_name[cid]
    for cid in train_class_count
    if val_class_count.get(cid, 0) == 0
]

if missing_in_val:
    print(f"  ⚠ val에 없는 클래스 ({len(missing_in_val)}개):")
    for name in missing_in_val:
        print(f"    - {name}")
else:
    print("  모든 클래스가 val에 포함됨")

print("\n" + "=" * 55)
print("변환 완료! 생성된 구조:")
print("=" * 55)
print(f"  {DATASET_DIR}/")
print(f"    images/train/  → {len(list((DATASET_DIR/'images'/'train').glob('*.png')))}장")
print(f"    images/val/    → {len(list((DATASET_DIR/'images'/'val').glob('*.png')))}장")
print(f"    labels/train/  → {len(list((DATASET_DIR/'labels'/'train').glob('*.txt')))}개")
print(f"    labels/val/    → {len(list((DATASET_DIR/'labels'/'val').glob('*.txt')))}개")
print(f"  {yaml_path}")

COCO JSON → YOLO 포맷 변환 시작

[1] Stratified train/val split 진행 중...
  Train 이미지: 186장
  Val   이미지: 46장

[2] YOLO txt 라벨 파일 생성 중...
  변환 완료: 232개
  스킵 (이미지 없음): 0개

[3] pill.yaml 생성 중...
  저장: /content/drive/MyDrive/ai_project/output/pill.yaml

[4] Split 분포 검증...
  모든 클래스가 val에 포함됨

변환 완료! 생성된 구조:
  /content/drive/MyDrive/ai_project/output/dataset/
    images/train/  → 186장
    images/val/    → 46장
    labels/train/  → 186개
    labels/val/    → 46개
  /content/drive/MyDrive/ai_project/output/pill.yaml


## 4️⃣ 데이터 증강

In [14]:
R"""
데이터 증강
------------------------------
- Albumentations 기반 공통 증강
- 모델 포맷과 무관하게 동작 (COCO bbox 기준)
- 증강 전/후 시각화 포함

설치: pip install albumentations Pillow
R"""

import json
import random
import zipfile
import io
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import defaultdict
from PIL import Image

import albumentations as A
from albumentations.pytorch import ToTensorV2

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
random.seed(42)
np.random.seed(42)

# 경로 설정
ZIP_PATH    = Path(ZIP_PATH)
OUTPUT_DIR = Path(OUTPUT_DIR)
COCO_JSON  = OUTPUT_DIR / "train_coco.json"
MAPPING_JSON = OUTPUT_DIR / "category_mapping.json"
AUG_DIR    = OUTPUT_DIR / "augmentation_samples"
AUG_DIR.mkdir(exist_ok=True)

with open(COCO_JSON, encoding="utf-8") as f:
    coco = json.load(f)
with open(MAPPING_JSON, encoding="utf-8") as f:
    mapping = json.load(f)

id_to_name = {int(k): v for k, v in mapping["id_to_name"].items()}
img_anns   = defaultdict(list)
for ann in coco["annotations"]:
    img_anns[ann["image_id"]].append(ann)
images = {img["id"]: img for img in coco["images"]}

# 증강 정의
def get_train_transform(img_size: int = 640) -> A.Compose:
    R"""
    학습용 증강 파이프라인
    bbox_params: COCO 포맷 [x_min, y_min, w, h] 절댓값 좌표
R"""
    return A.Compose([

        # 1. 기하학적 변환
        A.RandomRotate90(p=0.5),        # 90도 단위 회전
        A.Rotate(                        # 자유 회전 ±180도
            limit=180,
            border_mode=0,              # 빈 부분 검정으로 채움
            p=0.7
        ),
        A.HorizontalFlip(p=0.5),        # 좌우 반전
        A.VerticalFlip(p=0.5),          # 상하 반전

        # 2. 크기/위치 변환
        A.RandomResizedCrop(
            size=(img_size, img_size),
            scale=(0.7, 1.0),           # 70~100% 크기로 crop
            ratio=(0.9, 1.1),
            p=0.5
        ),
        A.ShiftScaleRotate(
            shift_limit=0.1,
            scale_limit=0.3,
            rotate_limit=0,             # 회전은 위에서 처리
            border_mode=0,
            p=0.4
        ),

        # 3. 색상 변환
        A.ColorJitter(
            brightness=0.3,
            contrast=0.3,
            saturation=0.4,
            hue=0.1,                    # 색조는 약하게 (알약 색상 보존)
            p=0.6
        ),
        A.HueSaturationValue(
            hue_shift_limit=10,
            sat_shift_limit=40,
            val_shift_limit=30,
            p=0.4
        ),

        # 4. 노이즈/품질
        A.GaussNoise(
            std_range=(0.01, 0.05),     # 약한 노이즈 (각인 보존)
            p=0.3
        ),
        A.ImageCompression(
            quality_range=(80, 100),    # JPEG 압축 시뮬레이션
            p=0.2
        ),

        # 5. 가리기 (Occlusion)
        A.CoarseDropout(
            num_holes_range=(1, 3),
            hole_height_range=(20, 60),
            hole_width_range=(20, 60),
            fill=0,                     # 검정으로 가림
            p=0.3
        ),

        # 6. 최종 리사이즈
        A.Resize(img_size, img_size),

    ],
    bbox_params=A.BboxParams(
        format='coco',                  # [x_min, y_min, w, h] 절댓값
        label_fields=['category_ids'],
        min_visibility=0.3,             # 30% 미만 가려진 bbox 제거
        clip=True,
    ))


def get_val_transform(img_size: int = 640) -> A.Compose:
    R"""검증/추론용 변환 — 리사이즈만R"""
    return A.Compose([
        A.Resize(img_size, img_size),
    ],
    bbox_params=A.BboxParams(
        format='coco',
        label_fields=['category_ids'],
        clip=True,
    ))


# 증강 적용 함수
def apply_augmentation(
    image: np.ndarray,
    bboxes: list,          # [[x,y,w,h], ...] COCO 절댓값
    category_ids: list,    # [cat_id, ...]
    transform: A.Compose,
) -> tuple:
    R"""
    증강 적용 후 (image, bboxes, category_ids) 반환
    bboxes: COCO 포맷 유지
R"""
    result = transform(
        image        = image,
        bboxes       = bboxes,
        category_ids = category_ids,
    )
    return (
        result["image"],
        list(result["bboxes"]),
        list(result["category_ids"]),
    )


# 증강 시각화
def visualize_augmentations(image_id: int, n_aug: int = 5):
    R"""원본 + 증강 n개를 나란히 시각화R"""
    img_info = images.get(image_id)
    if img_info is None:
        return

    anns = img_anns.get(image_id, [])
    bboxes      = [ann["bbox"] for ann in anns]
    category_ids= [ann["category_id"] for ann in anns]
    colors_bbox = ['#E24B4A', '#1D9E75', '#378ADD', '#EF9F27']

    # 이미지 로드
    fname = img_info["file_name"]
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        png_map = {Path(f).name: f for f in z.namelist()
                   if f.endswith(".png") and "train_images" in f}
        if fname not in png_map:
            print(f"  이미지 없음: {fname}")
            return
        with z.open(png_map[fname]) as f:
            img_pil = Image.open(io.BytesIO(f.read())).convert("RGB")

    img_np = np.array(img_pil)
    transform = get_train_transform(640)

    fig, axes = plt.subplots(1, n_aug + 1, figsize=(4 * (n_aug + 1), 5))

    def draw_boxes(ax, img, boxes, cat_ids, title):
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=8)
        h, w = img.shape[:2]
        for i, (box, cid) in enumerate(zip(boxes, cat_ids)):
            x, y, bw, bh = box
            # 정규화 여부 자동 판별
            if max(x, y, bw, bh) <= 1.0:
                x, y, bw, bh = x*w, y*h, bw*w, bh*h
            color = colors_bbox[i % len(colors_bbox)]
            rect = patches.Rectangle((x, y), bw, bh,
                                      linewidth=1.5, edgecolor=color,
                                      facecolor='none')
            ax.add_patch(rect)
            name = id_to_name.get(cid, str(cid))[:8]
            ax.text(x, max(0, y-3), name, color=color, fontsize=6,
                    bbox=dict(facecolor='white', alpha=0.6,
                              edgecolor='none', pad=1))

    # 원본
    draw_boxes(axes[0], img_np, bboxes, category_ids, "원본")

    # 증강 n개
    for idx in range(n_aug):
        try:
            aug_img, aug_boxes, aug_cats = apply_augmentation(
                img_np, bboxes, category_ids, transform)
            draw_boxes(axes[idx+1], aug_img, aug_boxes, aug_cats,
                       f"증강 {idx+1}")
        except Exception as e:
            axes[idx+1].set_title(f"증강 {idx+1}\n오류")
            axes[idx+1].axis('off')

    plt.suptitle(f"증강 전/후 비교\n{fname[:40]}", fontsize=10)
    plt.tight_layout()
    save_path = AUG_DIR / f"aug_sample_{image_id}.png"
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  저장: {save_path}")


# 실행
print("=" * 60)
print("증강 파이프라인 시각화")
print("=" * 60)

# 알약 4개짜리 이미지 3장 선택해서 시각화
sample_ids = [img["id"] for img in coco["images"]
              if len(img_anns.get(img["id"], [])) == 4][:3]

for img_id in sample_ids:
    print(f"\n  이미지 ID {img_id} 증강 시각화 중...")
    visualize_augmentations(img_id, n_aug=4)

print("\n" + "=" * 60)
print("증강 파이프라인 요약")
print("=" * 60)
print("  적용 증강 목록:")
print("    1. RandomRotate90 + Rotate(±180°) — 알약은 방향 무관")
print("    2. HorizontalFlip + VerticalFlip  — 대칭 알약 다양성")
print("    3. RandomResizedCrop              — 크기/위치 다양성")
print("    4. ColorJitter + HSV              — 배경/조명 다양성")
print("    5. GaussNoise                     — 실제 촬영 노이즈")
print("    6. CoarseDropout                  — 부분 가림 대응")
print(f"\n  시각화 저장: {AUG_DIR}")
print("\n  다음 단계: dataset_builder.py 실행")
print("  (모델별 포맷 변환: YOLO txt / COCO JSON)")
print("=" * 60)

# 모델별 사용 가이드 출력
print("\n" + "=" * 60)
print("모델별 증강 적용 방법")
print("=" * 60)
print("""
  [YOLOv11 / RT-DETRv2] — ultralytics 내장 증강 사용
    model.train(
        mosaic=1.0, degrees=180, flipud=0.5, fliplr=0.5,
        hsv_s=0.7, hsv_v=0.4, copy_paste=0.3
    )
    → pill.yaml만 있으면 됨. Albumentations 별도 불필요.

  [EfficientViT / DINO] — 커스텀 DataLoader에 적용
    transform = get_train_transform(640)
    aug_img, aug_boxes, aug_cats = apply_augmentation(
        image, bboxes, category_ids, transform
    )
    → 이 파일의 함수를 DataLoader의 __getitem__에서 호출.
""")

증강 파이프라인 시각화

  이미지 ID 1 증강 시각화 중...


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_2032/1745484410.py:228: UserWarning: Glyph 50896 (\N{HANGUL SYLLABLE WEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/1745484410.py:228: UserWarning: Glyph 48376 (\N{HANGUL SYLLABLE BON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/1745484410.py:228: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/1745484410.py:228: UserWarning: Glyph 47161 (\N{HANGUL SYLLABLE RYEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/1745484410.py:228: UserWarning: Glyph 48512 (\N{HANGUL SYLLABLE BU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2032/1745484410.py:228: UserWar

  저장: /content/drive/MyDrive/ai_project/output/augmentation_samples/aug_sample_1.png

  이미지 ID 2 증강 시각화 중...


  저장: /content/drive/MyDrive/ai_project/output/augmentation_samples/aug_sample_2.png

  이미지 ID 3 증강 시각화 중...


  저장: /content/drive/MyDrive/ai_project/output/augmentation_samples/aug_sample_3.png

증강 파이프라인 요약
  적용 증강 목록:
    1. RandomRotate90 + Rotate(±180°) — 알약은 방향 무관
    2. HorizontalFlip + VerticalFlip  — 대칭 알약 다양성
    3. RandomResizedCrop              — 크기/위치 다양성
    4. ColorJitter + HSV              — 배경/조명 다양성
    5. GaussNoise                     — 실제 촬영 노이즈
    6. CoarseDropout                  — 부분 가림 대응

  시각화 저장: /content/drive/MyDrive/ai_project/output/augmentation_samples

  다음 단계: dataset_builder.py 실행
  (모델별 포맷 변환: YOLO txt / COCO JSON)

모델별 증강 적용 방법

  [YOLOv11 / RT-DETRv2] — ultralytics 내장 증강 사용
    model.train(
        mosaic=1.0, degrees=180, flipud=0.5, fliplr=0.5,
        hsv_s=0.7, hsv_v=0.4, copy_paste=0.3
    )
    → pill.yaml만 있으면 됨. Albumentations 별도 불필요.

  [EfficientViT / DINO] — 커스텀 DataLoader에 적용
    transform = get_train_transform(640)
    aug_img, aug_boxes, aug_cats = apply_augmentation(
        image, bboxes, category_ids, transform
    )
    → 이 파일의 함수를 DataL